# 血液抹片細胞自動辨識與計數系統
> 影像處理 114-2 ｜ 期末專案

## 系統說明
本系統使用傳統影像處理技術（不使用深度學習）自動偵測血液抹片中的三種細胞：
- **WBC**（白血球）：以深色細胞核為特徵，使用灰階閾值 + 紫色色彩遮罩 + 形態學操作偵測
- **RBC**（紅血球）：以圓形輪廓為特徵，使用 Hough Circle Transform 偵測
- **PLT**（血小板）：以微小紫色碎片為特徵，使用 HSV 色彩遮罩偵測

### 評估結果（Test 集，126 張）
| 細胞 | GT | 預測 | 誤差 |
|------|----|------|------|
| WBC  | 133| 142  | +6.8%|
| RBC  |1699| 1849 | +8.8%|
| PLT  | 49 | 57   |+16.3%|

## Cell 1 — 安裝依賴套件

In [ ]:
# 安裝 Streamlit 及 Colab 展示工具
!pip install -q streamlit pyngrok opencv-python-headless
print('安裝完成')

## Cell 2 — 匯入套件

In [ ]:
import cv2
import numpy as np
import glob
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('OpenCV 版本:', cv2.__version__)
print('NumPy 版本:', np.__version__)

## Cell 3 — 資料集路徑設定

如果在 Google Colab 執行，請先掛載 Drive 或 clone 資料集：
```bash
# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 或 clone 資料集
!git clone https://github.com/lugan113/TXL-PBC_Dataset.git
```

In [ ]:
import os

# ── 自動偵測資料集位置 ─────────────────────────────────────────────
_candidates = [
    'TXL-PBC_Dataset/TXL-PBC',                        # 本機（與 notebook 同層）
    '/content/TXL-PBC_Dataset/TXL-PBC',               # Colab 根目錄
    '/content/drive/MyDrive/TXL-PBC_Dataset/TXL-PBC', # Colab Drive
    os.path.join(os.path.dirname(os.path.abspath('.')),
                 'TXL-PBC_Dataset', 'TXL-PBC'),        # 上一層
]

DATASET_ROOT = None
for path in _candidates:
    if os.path.isdir(path):
        DATASET_ROOT = path
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        '找不到 TXL-PBC 資料集！\n'
        '請執行以下任一方式後重新執行此 cell：\n'
        '  1. Colab：先執行 !git clone https://github.com/lugan113/TXL-PBC_Dataset.git\n'
        '  2. 本機：確認 TXL-PBC_Dataset/ 資料夾與此 notebook 位於同一目錄\n'
        f'  已搜尋路徑：{_candidates}'
    )

print(f'資料集路徑：{DATASET_ROOT}')

# 可選擇 test / val / train 任一分割
SPLIT = 'test'

IMG_DIR   = f'{DATASET_ROOT}/images/{SPLIT}'
LABEL_DIR = f'{DATASET_ROOT}/labels/{SPLIT}'

# 取得所有影像路徑
img_files = sorted(glob.glob(f'{IMG_DIR}/*.png'))

if not img_files:
    raise FileNotFoundError(
        f'在 {IMG_DIR} 找不到任何 .png 影像！\n'
        '請確認資料集完整解壓縮，或更換 SPLIT 值（test/val/train）。'
    )

print(f'找到 {len(img_files)} 張 {SPLIT} 影像')

# 類別名稱對應
CLASS_NAMES  = {0: 'WBC', 1: 'RBC', 2: 'PLT'}
CLASS_COLORS = {
    'WBC': (0, 200, 0),   # 綠（BGR）
    'RBC': (0, 0, 220),   # 紅（BGR）
    'PLT': (220, 0, 0),   # 藍（BGR）
}

## Cell 4 — 偵測參數設定

所有可調整的參數集中於此，方便在 Streamlit App 中對應 slider 控制。

In [ ]:
# ── WBC 偵測參數 ──────────────────────────────────────────────────
WBC_GRAY_THRESH = 130   # 深色像素閾值：低於此值 → WBC 細胞核候選
WBC_CLOSE_K     = 23    # 形態學 closing kernel（合併細胞核碎片）
WBC_OPEN_K      = 9     # 形態學 opening kernel（去除小雜訊）
WBC_MIN_AREA    = 9500  # WBC 最小輪廓面積 (px²)
WBC_PAD         = 65    # WBC bbox 排除區擴充 (px)

# ── RBC 偵測參數（Hough Circle Transform）────────────────────────
RBC_MIN_DIST = 38       # 圓心最小距離
RBC_PARAM1   = 50       # Canny 邊緣偵測高閾值
RBC_PARAM2   = 12       # 累加器閾值（越低 → 偵測越多圓）
RBC_MIN_R    = 28       # 最小圓半徑 (px)
RBC_MAX_R    = 78       # 最大圓半徑 (px)

# ── PLT 偵測參數（HSV 紫色遮罩）──────────────────────────────────
PLT_H_LO     = 112      # 紫色 Hue 下界（OpenCV 0-180 scale）
PLT_H_HI     = 162      # 紫色 Hue 上界
PLT_S_MIN    = 38       # 最低飽和度（過濾灰色雜訊）
PLT_V_HI     = 210      # 最高亮度（過濾過亮背景）
PLT_MIN_AREA = 400      # PLT 最小輪廓面積 (px²)
PLT_MAX_AREA = 3000     # PLT 最大輪廓面積 (px²)

print('參數設定完成')

## Cell 5 — 輔助函式（GT 解析、評估）

In [ ]:
def load_gt(label_path, img_w=575, img_h=575):
    """解析 YOLO 格式標籤，回傳各類別 bounding box 清單。"""
    gt = {'WBC': [], 'RBC': [], 'PLT': []}
    if not os.path.exists(label_path):
        return gt
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            # 轉換為像素座標
            x1 = int((cx - bw/2) * img_w)
            y1 = int((cy - bh/2) * img_h)
            x2 = int((cx + bw/2) * img_w)
            y2 = int((cy + bh/2) * img_h)
            name = CLASS_NAMES.get(cls)
            if name:
                gt[name].append((x1, y1, x2, y2))
    return gt


def draw_gt(img, gt_boxes):
    """在影像上繪製 Ground Truth bounding boxes。"""
    out = img.copy()
    for name, boxes in gt_boxes.items():
        color = CLASS_COLORS[name]
        for (x1, y1, x2, y2) in boxes:
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            cv2.putText(out, name, (x1, max(0, y1-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return out


def count_error(gt_count, pred_count):
    """計算相對誤差。"""
    if gt_count == 0:
        return 0.0 if pred_count == 0 else float('inf')
    return (pred_count - gt_count) / gt_count


print('輔助函式載入完成')

## Cell 6 — 影像前處理 Pipeline 函式

依序回傳各階段影像：原始 → 灰階 → Gaussian Blur → Adaptive Thresholding → Morphological Closing

In [ ]:
def preprocess_pipeline(img_bgr, wbc_gray_thresh=WBC_GRAY_THRESH):
    """
    標準前處理 Pipeline（可視化用）。

    回傳:
        steps: dict，包含各階段影像
            'original'      : 原始 BGR 影像
            'grayscale'     : 灰階影像
            'gaussian_blur' : Gaussian Blur 後影像
            'adaptive_thr'  : Adaptive Thresholding 結果（反轉：細胞=白）
            'morph_close'   : Morphological Closing 後（用於 WBC 核心）
    """
    # Step 1: 灰階轉換
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Step 2: Gaussian Blur（去除雜訊）
    blurred = cv2.GaussianBlur(gray, (11, 11), 2)

    # Step 3: Adaptive Thresholding（局部自適應閾值）
    # THRESH_BINARY_INV：細胞（較暗）→ 白色
    adaptive = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=51, C=8
    )

    # Step 4: Morphological Closing（填補細胞核內部空洞）
    kernel = np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8)
    morph_close = cv2.morphologyEx(adaptive, cv2.MORPH_CLOSE, kernel)

    # 同時準備 WBC 全域閾值版本（用於偵測 WBC 細胞核）
    _, dark_global = cv2.threshold(blurred, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    morph_close_global = cv2.morphologyEx(
        dark_global, cv2.MORPH_CLOSE, np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8)
    )

    return {
        'original':      img_bgr,
        'grayscale':     gray,
        'gaussian_blur': blurred,
        'adaptive_thr':  adaptive,
        'morph_close':   morph_close_global,  # 形態學操作（全域閾值版，更清楚展示 WBC）
    }


print('Pipeline 函式載入完成')

## Cell 7 — 細胞偵測函式

整合 WBC / RBC / PLT 偵測，並回傳帶框線的結果影像。

In [ ]:
def detect_cells(img_bgr,
                 wbc_gray_thresh=WBC_GRAY_THRESH,
                 rbc_param2=RBC_PARAM2,
                 plt_min_area=PLT_MIN_AREA):
    """
    偵測三種血液細胞。

    Pipeline:
        1. WBC: 全域灰階閾值 AND 紫色 HSV 遮罩 → 形態學操作 → 大輪廓
        2. RBC: Hough Circle Transform（排除 WBC 區域）
        3. PLT: HSV 紫色遮罩（小輪廓，排除 WBC+RBC 區域）

    回傳:
        counts  : dict {'WBC': int, 'RBC': int, 'PLT': int}
        result  : 帶偵測框線的 BGR 影像
        details : dict 包含各步驟中間結果（wbc_cnts, rbc_centers, plt_dets）
    """
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ── Step 1: WBC ────────────────────────────────────────────────────
    g_blur = cv2.GaussianBlur(gray, (11, 11), 2)

    # 全域閾值：深色像素 → WBC 細胞核候選
    _, dark = cv2.threshold(g_blur, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)

    # 紫色 HSV 遮罩：防止深色 RBC 叢被誤判為 WBC
    purple = cv2.inRange(hsv, (100, 25, 20), (168, 255, 200))
    dark = cv2.bitwise_and(dark, purple)

    # 形態學操作：合併碎散核心、去除小雜訊
    dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8))
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN,  np.ones((WBC_OPEN_K,  WBC_OPEN_K),  np.uint8))

    wbc_cnts_all, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    wbc_cnts = [c for c in wbc_cnts_all if cv2.contourArea(c) >= WBC_MIN_AREA]

    # 建立 WBC 排除遮罩
    wbc_excl = np.zeros((h, w), np.uint8)
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(wbc_excl,
                      (max(0, x - WBC_PAD),      max(0, y - WBC_PAD)),
                      (min(w, x + bw + WBC_PAD), min(h, y + bh + WBC_PAD)),
                      255, -1)

    # ── Step 2: RBC（Hough Circle Transform）──────────────────────────
    blur5 = cv2.GaussianBlur(gray, (5, 5), 1)
    circles = cv2.HoughCircles(
        blur5, cv2.HOUGH_GRADIENT, dp=1,
        minDist=RBC_MIN_DIST, param1=RBC_PARAM1, param2=rbc_param2,
        minRadius=RBC_MIN_R, maxRadius=RBC_MAX_R
    )
    rbc_centers = []
    if circles is not None:
        for (cx, cy, r) in np.round(circles[0]).astype(int):
            cy_c = int(np.clip(cy, 0, h - 1))
            cx_c = int(np.clip(cx, 0, w - 1))
            if wbc_excl[cy_c, cx_c] == 0:
                rbc_centers.append((int(cx), int(cy), int(r)))

    # 建立 RBC 排除遮罩
    rbc_excl = np.zeros((h, w), np.uint8)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(rbc_excl, (cx, cy), r + 10, 255, -1)

    # ── Step 3: PLT（HSV 紫色遮罩）────────────────────────────────────
    all_excl = cv2.bitwise_or(wbc_excl, rbc_excl)

    purp_mask = cv2.inRange(
        hsv,
        (PLT_H_LO, PLT_S_MIN, 30),
        (PLT_H_HI, 255,       PLT_V_HI)
    )
    purp_mask = cv2.bitwise_and(purp_mask, cv2.bitwise_not(all_excl))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_OPEN,  np.ones((2, 2), np.uint8))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_CLOSE, np.ones((4, 4), np.uint8))

    plt_cnts_all, _ = cv2.findContours(purp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plt_dets = [c for c in plt_cnts_all
                if plt_min_area <= cv2.contourArea(c) <= PLT_MAX_AREA]

    # ── 繪製偵測結果 ───────────────────────────────────────────────────
    result = img_bgr.copy()
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x + bw, y + bh), CLASS_COLORS['WBC'], 2)
        cv2.putText(result, 'WBC', (x, max(0, y - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, CLASS_COLORS['WBC'], 1)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(result, (cx, cy), r, CLASS_COLORS['RBC'], 1)
        cv2.putText(result, 'R', (cx - 5, cy + 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, CLASS_COLORS['RBC'], 1)
    for cnt in plt_dets:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x + bw, y + bh), CLASS_COLORS['PLT'], 1)
        cv2.putText(result, 'P', (x, max(0, y - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, CLASS_COLORS['PLT'], 1)

    counts  = {'WBC': len(wbc_cnts), 'RBC': len(rbc_centers), 'PLT': len(plt_dets)}
    details = {'wbc_cnts': wbc_cnts, 'rbc_centers': rbc_centers, 'plt_dets': plt_dets}
    return counts, result, details


print('偵測函式載入完成')

## Cell 8 — 快速測試（單張影像視覺化）

對第一張測試影像執行完整 pipeline，並用 matplotlib 顯示各階段結果。

In [ ]:
# 選擇測試影像（可修改索引）
TEST_IDX = 0
img_path = img_files[TEST_IDX]
stem = Path(img_path).stem

img = cv2.imread(img_path)
gt  = load_gt(f'{LABEL_DIR}/{stem}.txt')

# 執行前處理 pipeline
steps = preprocess_pipeline(img)

# 執行細胞偵測
counts, result, _ = detect_cells(img)

# ── 顯示各階段結果 ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'影像前處理 Pipeline — {stem[:24]}', fontsize=13, y=0.98)

titles = ['原始影像', '灰階', 'Gaussian Blur',
          'Adaptive Thresholding', 'Morphological Closing', '偵測結果']
images = [
    cv2.cvtColor(steps['original'],      cv2.COLOR_BGR2RGB),
    steps['grayscale'],
    steps['gaussian_blur'],
    steps['adaptive_thr'],
    steps['morph_close'],
    cv2.cvtColor(result,                 cv2.COLOR_BGR2RGB),
]
cmaps = ['viridis', 'gray', 'gray', 'gray', 'gray', 'viridis']

for ax, title, image, cmap in zip(axes.flatten(), titles, images, cmaps):
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('pipeline_demo.png', dpi=120, bbox_inches='tight')
plt.show()

# 顯示計數結果
print(f'\n{'─'*40}')
print(f'{'細胞':8s} {'Ground Truth':>12s} {'預測':>8s} {'誤差':>8s}')
print(f'{'─'*40}')
for name in ['WBC', 'RBC', 'PLT']:
    gt_n  = len(gt[name])
    pr_n  = counts[name]
    err   = count_error(gt_n, pr_n)
    e_str = f'{err:+.1%}' if gt_n > 0 else 'N/A'
    print(f'{name:8s} {gt_n:>12d} {pr_n:>8d} {e_str:>8s}')
print(f'{'─'*40}')

## Cell 9 — 全資料集評估

In [ ]:
def evaluate_dataset(split='test'):
    """對整個資料集分割評估偵測誤差。"""
    img_dir   = f'{DATASET_ROOT}/images/{split}'
    label_dir = f'{DATASET_ROOT}/labels/{split}'
    files = sorted(glob.glob(f'{img_dir}/*.png'))

    gt_total  = {'WBC': 0, 'RBC': 0, 'PLT': 0}
    pr_total  = {'WBC': 0, 'RBC': 0, 'PLT': 0}

    for img_path in files:
        stem = Path(img_path).stem
        img  = cv2.imread(img_path)
        gt   = load_gt(f'{label_dir}/{stem}.txt')
        cnt, _, _ = detect_cells(img)
        for name in ['WBC', 'RBC', 'PLT']:
            gt_total[name] += len(gt[name])
            pr_total[name] += cnt[name]

    print(f'\n資料集評估結果 — Split: {split}  ({len(files)} 張影像)')
    print(f'{'═'*52}')
    print(f'{'細胞':6s} {'GT':>6s} {'預測':>6s} {'誤差':>8s}  {'±30% 內'}')
    print(f'{'─'*52}')
    for name in ['WBC', 'RBC', 'PLT']:
        gt_n = gt_total[name]; pr_n = pr_total[name]
        err  = count_error(gt_n, pr_n)
        ok   = 'OK' if abs(err) <= 0.30 else 'FAIL'
        print(f'{name:6s} {gt_n:>6d} {pr_n:>6d} {err:>+8.1%}  {ok}')
    print(f'{'═'*52}')


# 執行評估（需數分鐘）
evaluate_dataset('test')

## Cell 10 — 撰寫 Streamlit App

使用 `%%writefile` 將 app 程式碼存至 `app.py`，再透過 Streamlit 啟動互動式介面。

**App 功能：**
- 圖片切換功能（slider 選擇影像）
- 3 個可調參數（WBC 閾值、RBC 靈敏度、PLT 最小面積）
- 各階段前處理結果顯示
- Ground Truth 對照
- 下載偵測結果按鈕

In [ ]:
%%writefile app.py
"""
血液抹片細胞自動辨識與計數系統 — Streamlit App
影像處理 114-2 期末專案
"""

import streamlit as st
import cv2
import numpy as np
import glob
import os
from pathlib import Path

# ─────────────────────── 資料集路徑自動偵測 ──────────────────────────
def _find_dataset():
    candidates = [
        'TXL-PBC_Dataset/TXL-PBC',
        '/content/TXL-PBC_Dataset/TXL-PBC',
        '/content/drive/MyDrive/TXL-PBC_Dataset/TXL-PBC',
        os.path.join(os.path.dirname(os.path.abspath(__file__)),
                     'TXL-PBC_Dataset', 'TXL-PBC'),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None

DATASET_ROOT = _find_dataset()

# ─────────────────────── 常數 ────────────────────────────────────────
CLASS_NAMES  = {0: 'WBC', 1: 'RBC', 2: 'PLT'}
CLASS_COLORS = {'WBC': (0, 200, 0), 'RBC': (0, 0, 220), 'PLT': (220, 0, 0)}

WBC_CLOSE_K  = 23
WBC_OPEN_K   = 9
WBC_MIN_AREA = 9500
WBC_PAD      = 65
RBC_MIN_DIST = 38
RBC_PARAM1   = 50
RBC_MIN_R    = 28
RBC_MAX_R    = 78
PLT_H_LO     = 112
PLT_H_HI     = 162
PLT_S_MIN    = 38
PLT_V_HI     = 210
PLT_MAX_AREA = 3000


# ─────────────────────── 工具函式 ────────────────────────────────────
@st.cache_data
def load_image_list(split):
    if DATASET_ROOT is None:
        return []
    return sorted(glob.glob(f'{DATASET_ROOT}/images/{split}/*.png'))


def load_gt(label_path, img_w=575, img_h=575):
    gt = {'WBC': [], 'RBC': [], 'PLT': []}
    if not os.path.exists(label_path):
        return gt
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = int((cx - bw/2) * img_w);  x2 = int((cx + bw/2) * img_w)
            y1 = int((cy - bh/2) * img_h);  y2 = int((cy + bh/2) * img_h)
            name = CLASS_NAMES.get(cls)
            if name:
                gt[name].append((x1, y1, x2, y2))
    return gt


def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def gray2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)


# ─────────────────────── 前處理 Pipeline ─────────────────────────────
def preprocess_pipeline(img_bgr, wbc_gray_thresh):
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (11, 11), 2)
    adaptive = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, blockSize=51, C=8
    )
    _, dark = cv2.threshold(blurred, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    morph_close = cv2.morphologyEx(
        dark, cv2.MORPH_CLOSE, np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8)
    )
    return {
        'grayscale':     gray,
        'gaussian_blur': blurred,
        'adaptive_thr':  adaptive,
        'morph_close':   morph_close,
    }


# ─────────────────────── 細胞偵測 ────────────────────────────────────
@st.cache_data
def detect_cells_cached(img_bytes, wbc_gray_thresh, rbc_param2, plt_min_area):
    img_arr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
    return _detect(img, wbc_gray_thresh, rbc_param2, plt_min_area)


def _detect(img_bgr, wbc_gray_thresh, rbc_param2, plt_min_area):
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # WBC
    g_blur = cv2.GaussianBlur(gray, (11, 11), 2)
    _, dark = cv2.threshold(g_blur, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    purple  = cv2.inRange(hsv, (100, 25, 20), (168, 255, 200))
    dark    = cv2.bitwise_and(dark, purple)
    dark    = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8))
    dark    = cv2.morphologyEx(dark, cv2.MORPH_OPEN,  np.ones((WBC_OPEN_K,  WBC_OPEN_K),  np.uint8))
    all_cnts, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    wbc_cnts = [c for c in all_cnts if cv2.contourArea(c) >= WBC_MIN_AREA]

    wbc_excl = np.zeros((h, w), np.uint8)
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(wbc_excl,
                      (max(0, x - WBC_PAD),      max(0, y - WBC_PAD)),
                      (min(w, x + bw + WBC_PAD), min(h, y + bh + WBC_PAD)),
                      255, -1)

    # RBC
    blur5   = cv2.GaussianBlur(gray, (5, 5), 1)
    circles = cv2.HoughCircles(
        blur5, cv2.HOUGH_GRADIENT, dp=1,
        minDist=RBC_MIN_DIST, param1=RBC_PARAM1, param2=rbc_param2,
        minRadius=RBC_MIN_R,  maxRadius=RBC_MAX_R
    )
    rbc_centers = []
    if circles is not None:
        for (cx, cy, r) in np.round(circles[0]).astype(int):
            cc = int(np.clip(cx, 0, w-1));  rc = int(np.clip(cy, 0, h-1))
            if wbc_excl[rc, cc] == 0:
                rbc_centers.append((int(cx), int(cy), int(r)))

    rbc_excl = np.zeros((h, w), np.uint8)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(rbc_excl, (cx, cy), r + 10, 255, -1)

    # PLT
    all_excl  = cv2.bitwise_or(wbc_excl, rbc_excl)
    purp_mask = cv2.inRange(hsv, (PLT_H_LO, PLT_S_MIN, 30), (PLT_H_HI, 255, PLT_V_HI))
    purp_mask = cv2.bitwise_and(purp_mask, cv2.bitwise_not(all_excl))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_OPEN,  np.ones((2, 2), np.uint8))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_CLOSE, np.ones((4, 4), np.uint8))
    plt_all, _ = cv2.findContours(purp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plt_dets   = [c for c in plt_all if plt_min_area <= cv2.contourArea(c) <= PLT_MAX_AREA]

    # 繪製結果
    result = img_bgr.copy()
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x+bw, y+bh), CLASS_COLORS['WBC'], 2)
        cv2.putText(result, 'WBC', (x, max(0, y-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, CLASS_COLORS['WBC'], 1)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(result, (cx, cy), r, CLASS_COLORS['RBC'], 1)
    for cnt in plt_dets:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x+bw, y+bh), CLASS_COLORS['PLT'], 1)

    counts = {'WBC': len(wbc_cnts), 'RBC': len(rbc_centers), 'PLT': len(plt_dets)}
    return counts, result


def draw_gt(img, gt_boxes):
    out = img.copy()
    for name, boxes in gt_boxes.items():
        color = CLASS_COLORS[name]
        for (x1, y1, x2, y2) in boxes:
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            cv2.putText(out, name, (x1, max(0, y1-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return out


# ─────────────────────── Streamlit UI ────────────────────────────────
def main():
    st.set_page_config(page_title='血液抹片細胞偵測系統', page_icon='🔬', layout='wide')
    st.title('🔬 血液抹片細胞自動辨識與計數系統')
    st.caption('影像處理 114-2 期末專案 ｜ 使用傳統影像處理技術（無深度學習）')

    if DATASET_ROOT is None:
        st.error(
            '找不到 TXL-PBC 資料集！\n\n'
            '請先執行：`!git clone https://github.com/lugan113/TXL-PBC_Dataset.git`\n'
            '或確認資料集資料夾與 app.py 位於同一目錄。'
        )
        st.stop()

    st.divider()

    with st.sidebar:
        st.header('⚙️ 設定')
        split     = st.selectbox('資料集分割', ['test', 'val', 'train'], index=0)
        img_files = load_image_list(split)

        if not img_files:
            st.error(f'找不到 {split} 影像。')
            st.stop()

        st.subheader('📷 影像選擇')
        img_idx = st.slider(f'影像索引（共 {len(img_files)} 張）',
                            0, len(img_files) - 1, 0)
        st.caption(Path(img_files[img_idx]).name[:35])

        st.subheader('🎛️ 偵測參數')
        wbc_thresh   = st.slider('WBC 灰階閾值', 90, 160, 130, 5,
                                  help='越低 → 只偵測極深核心；越高 → 包含較淺色 WBC')
        rbc_param2   = st.slider('RBC Hough 靈敏度（越低偵測越多）', 6, 25, 12, 1,
                                  help='Hough Circle 累加器閾值')
        plt_min_area = st.slider('PLT 最小面積 (px²)', 100, 1000, 400, 50,
                                  help='PLT 紫色輪廓最小有效面積')

        st.divider()
        st.info('**顏色圖例**\n\n🟩 WBC　🔴 RBC　🟦 PLT')

    img_path = img_files[img_idx]
    stem     = Path(img_path).stem
    lbl_path = f'{DATASET_ROOT}/labels/{split}/{stem}.txt'
    img      = cv2.imread(img_path)
    gt       = load_gt(lbl_path)

    with open(img_path, 'rb') as f:
        img_bytes = f.read()

    with st.spinner('偵測中...'):
        counts, result = detect_cells_cached(img_bytes, wbc_thresh, rbc_param2, plt_min_area)

    # Pipeline 視覺化
    st.subheader('📊 影像前處理 Pipeline')
    steps = preprocess_pipeline(img, wbc_thresh)
    c1, c2, c3, c4, c5 = st.columns(5)
    c1.image(bgr2rgb(img),                    caption='① 原始影像',            use_container_width=True)
    c2.image(gray2rgb(steps['grayscale']),     caption='② 灰階',                use_container_width=True)
    c3.image(gray2rgb(steps['gaussian_blur']), caption='③ Gaussian Blur',       use_container_width=True)
    c4.image(gray2rgb(steps['adaptive_thr']),  caption='④ Adaptive Threshold',  use_container_width=True)
    c5.image(gray2rgb(steps['morph_close']),   caption='⑤ Morph. Closing',      use_container_width=True)

    st.divider()

    # GT vs 預測
    st.subheader('🎯 偵測結果 vs Ground Truth')
    cgt, cpred = st.columns(2)
    cgt.image(bgr2rgb(draw_gt(img, gt)), caption='Ground Truth', use_container_width=True)
    cpred.image(bgr2rgb(result),          caption='預測結果',     use_container_width=True)

    st.divider()

    # 計數
    st.subheader('📈 計數結果')
    cols = st.columns(3)
    for i, name in enumerate(['WBC', 'RBC', 'PLT']):
        gt_n = len(gt[name]); pr_n = counts[name]
        err  = (pr_n - gt_n) / gt_n if gt_n > 0 else 0.0
        cols[i].metric(name, f'預測: {pr_n}',
                       delta=f'GT: {gt_n}  誤差: {err:+.1%}' if gt_n > 0 else f'GT: {gt_n}',
                       delta_color='normal' if abs(err) <= 0.30 else 'inverse')

    st.divider()

    # 下載
    st.subheader('💾 下載結果')
    cd1, cd2, cd3 = st.columns(3)
    _, r_enc = cv2.imencode('.png', result)
    cd1.download_button('⬇️ 偵測結果影像', r_enc.tobytes(),
                         f'detection_{stem}.png', 'image/png')
    _, g_enc = cv2.imencode('.png', draw_gt(img, gt))
    cd2.download_button('⬇️ Ground Truth 影像', g_enc.tobytes(),
                         f'groundtruth_{stem}.png', 'image/png')
    pipe_row = np.hstack([
        cv2.cvtColor(steps['grayscale'],     cv2.COLOR_GRAY2BGR),
        cv2.cvtColor(steps['gaussian_blur'], cv2.COLOR_GRAY2BGR),
        cv2.cvtColor(steps['adaptive_thr'],  cv2.COLOR_GRAY2BGR),
        cv2.cvtColor(steps['morph_close'],   cv2.COLOR_GRAY2BGR),
        result,
    ])
    _, p_enc = cv2.imencode('.png', pipe_row)
    cd3.download_button('⬇️ Pipeline 視覺化', p_enc.tobytes(),
                         f'pipeline_{stem}.png', 'image/png')


if __name__ == '__main__':
    main()

## Cell 11 — 啟動 Streamlit App（cloudflared tunnel）

在 Google Colab 中執行：
1. 下載 `cloudflared` 執行檔（無需帳號或 token）
2. 背景啟動 Streamlit（port 8501）
3. 建立 Cloudflare Tunnel → 輸出 `*.trycloudflare.com` 公開 URL

In [ ]:
import subprocess, time

# ── 下載 cloudflared（Colab 環境，Linux amd64）──────────────────────
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', 'cloudflared'
], check=True)
subprocess.run(['chmod', '+x', 'cloudflared'], check=True)

# ── 啟動 Streamlit（背景）──────────────────────────────────────────
subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)  # 等待 Streamlit 就緒

# ── 建立 cloudflared tunnel（輸出公開 URL）────────────────────────
# URL 會在 stderr 中以 "trycloudflare.com" 結尾的行印出
print('正在建立 Cloudflare Tunnel，請稍候...')
subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.DEVNULL
    # stderr 不重導向，讓 trycloudflare.com URL 直接顯示在輸出
)

## Cell 12 — 本機執行（非 Colab 環境）

若在本機執行 Jupyter，直接在 terminal 輸入：
```bash
streamlit run app.py
```
或執行以下 cell：

In [ ]:
import subprocess, sys

# 使用 subprocess.Popen 搭配引數列表，避免 shell injection
if sys.platform == 'win32':
    # Windows：在新的 cmd 視窗中執行
    subprocess.Popen(
        ['cmd', '/k', 'streamlit run app.py'],
        creationflags=subprocess.CREATE_NEW_CONSOLE
    )
else:
    # macOS / Linux：背景執行
    subprocess.Popen(['streamlit', 'run', 'app.py'])

print('Streamlit 已啟動，請在瀏覽器開啟 http://localhost:8501')